# 🦅 Falcon-7B-Instruct — Medical QA Fine-Tuning Pipeline

**One notebook. Run top to bottom. That's it.**

All logic lives in `src/` Python modules. This notebook just calls them in order.

| Step | What happens | Module |
|------|-------------|--------|
| 1 | Install deps, clone repo | — |
| 2 | Load config + verify GPU | `src/utils.py` |
| 3 | Load tokenizer + model (4-bit) | `src/model.py` |
| 4 | Load & tokenize MedQA dataset | `src/data.py` |
| 5 | Apply LoRA, train | `src/train.py` |
| 6 | TensorBoard | — |
| 7 | Evaluate: PPL · MC Accuracy · ROUGE | `src/evaluate.py` |
| 8 | Qualitative examples | `src/evaluate.py` |
| 9 | Plot training curves | — |
| 10 | Download artifacts | — |

> ⚡ **Before running:** Runtime → Change runtime type → **GPU (T4 / P100)**
>
> 💾 **All outputs save directly to `/kaggle/working/` — the only persistent directory in Kaggle.**

---
## Step 1 — Install Dependencies & Clone Repo

In [ ]:
import os, sys
from pathlib import Path

# ── Detect Kaggle environment ─────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()
PERSISTENT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("./outputs")
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"  Platform  : {'Kaggle ✅' if ON_KAGGLE else 'Local'}")
print(f"  Persistent: {PERSISTENT_DIR}")

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/singularity-14/finetune-falcon-7b-instruct.git"
REPO_DIR = "finetune-falcon-7b-instruct"

# Clone INTO /kaggle/working/ so src/ is accessible
if ON_KAGGLE:
    clone_path = Path("/kaggle/working") / REPO_DIR
else:
    clone_path = Path(REPO_DIR)

if not clone_path.exists():
    print(f"Cloning repo → {clone_path}")
    os.system(f"git clone {REPO_URL} {clone_path}")
else:
    print(f"📁 Repo already exists at {clone_path}")

# Change working dir to repo root
os.chdir(clone_path)
sys.path.insert(0, str(clone_path))   # make src/ importable
print(f"  CWD       : {os.getcwd()}")

# ── Install deps ──────────────────────────────────────────────────────────────
os.system("pip install -q -r requirements-colab.txt")
print("\n✅ Dependencies installed")

---
## Step 2 — Load Config & Verify GPU

In [ ]:
from src.utils import load_config, setup_logging, set_seed, gpu_report, make_output_dirs, free_memory

logger    = setup_logging()
cfg       = load_config("configs/config.yaml")
set_seed(cfg["dataset"]["seed"])

# ── Override output paths to /kaggle/working/ if on Kaggle ───────────────────
# This is the ONLY persistent directory in Kaggle.
# All other paths are wiped on session reset.
if ON_KAGGLE:
    KAGGLE_WD = "/kaggle/working"
    cfg["training"]["output_dir"]     = f"{KAGGLE_WD}/results"
    cfg["training"]["logging_dir"]    = f"{KAGGLE_WD}/logs"
    cfg["saving"]["final_model_dir"]  = f"{KAGGLE_WD}/final_model"
    cfg["saving"]["peft_adapter_dir"] = f"{KAGGLE_WD}/peft_adapter"
    print("  📌 Kaggle detected — outputs redirected to /kaggle/working/")

make_output_dirs(cfg)

print(f"\n  Model      : {cfg['model']['name']}")
print(f"  Dataset    : {cfg['dataset']['name']}")
print(f"  Train/Eval : {cfg['dataset']['train_samples']} / {cfg['dataset']['eval_samples']}")
print(f"  LoRA rank  : {cfg['lora']['r']}")
print(f"  Epochs     : {cfg['training']['num_train_epochs']} (max)")
print(f"  Output dir : {cfg['training']['output_dir']}")
print(f"  Model dir  : {cfg['saving']['peft_adapter_dir']}")

gpu_stats = gpu_report()

---
## Step 3 — Load Model with 4-bit Quantization + LoRA

In [ ]:
from src.model import load_tokenizer, build_quant_config, load_base_model, apply_lora

tokenizer    = load_tokenizer(cfg["model"]["name"], cfg["model"]["trust_remote_code"])
quant_config = build_quant_config(cfg)
model        = load_base_model(
    cfg["model"]["name"], quant_config, tokenizer,
    cfg["model"]["trust_remote_code"], cfg["model"]["use_cache"],
)
model = apply_lora(model, cfg)

---
## Step 4 — Load & Tokenize MedQA Dataset

In [ ]:
from src.data import load_and_split

train_dataset, eval_dataset, data_collator = load_and_split(cfg, tokenizer)

---
## Step 5 — Train

> 💾 Checkpoints save to `/kaggle/working/results/` every epoch.
> Even if this cell crashes, checkpoints survive the session reset.

In [ ]:
from src.train import build_trainer, run_training, save_artifacts

trainer       = build_trainer(model, cfg, train_dataset, eval_dataset, data_collator)
train_metrics = run_training(trainer, cfg)
final_eval    = save_artifacts(trainer, model, tokenizer, cfg)

# ── Verify files saved to /kaggle/working/ ────────────────────────────────────
print("\n📋 Verifying /kaggle/working/ contents:")
for p in sorted(Path("/kaggle/working").iterdir()):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**2 if p.is_dir() else p.stat().st_size / 1024**2
    print(f"  {'📁' if p.is_dir() else '📄'} {p.name}  ({size:.1f} MB)")

---
## Step 6 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {cfg['training']['logging_dir']}

---
## Step 7 — Evaluate

In [ ]:
from datasets import load_dataset
from src.evaluate import (
    compute_perplexity, compute_mc_accuracy,
    compute_rouge, save_and_print_summary,
)
import torch

DEVICE = "cuda"

raw        = load_dataset(cfg["dataset"]["name"], split=cfg["dataset"]["split"])
test_split = raw.train_test_split(test_size=cfg["dataset"]["test_size"], seed=cfg["dataset"]["seed"])
N          = cfg["evaluation"]["num_samples"]
eval_raw   = test_split["test"].select(range(N))

print(f"Evaluating on {N} held-out examples...\n")

perplexity    = compute_perplexity(model, tokenizer, eval_raw, cfg, DEVICE)
mc_results    = compute_mc_accuracy(model, tokenizer, eval_raw, cfg, DEVICE)
rouge_results = compute_rouge(model, tokenizer, eval_raw.select(range(min(20, N))), cfg, DEVICE)
summary       = save_and_print_summary(perplexity, mc_results, rouge_results, cfg)

---
## Step 8 — Qualitative Examples

In [ ]:
examples = rouge_results["examples"]
print("🔍 QUALITATIVE EXAMPLES\n" + "═" * 70)
for i, ex in enumerate(examples[:5]):
    print(f"\n[{i+1}] Question  : {ex['question'][:180]}...")
    print(f"     True Ans  : {ex['true']}")
    print(f"     Generated : {ex['generated'][:200]}")
    print(f"     ROUGE-1   : {ex['rouge1']:.3f}  |  ROUGE-L : {ex['rougeL']:.3f}")
    print("─" * 70)

---
## Step 9 — Plot Training Curves

In [ ]:
import json, math
import matplotlib.pyplot as plt

results_dir = Path(cfg["training"]["output_dir"])
state_files = sorted(results_dir.glob("checkpoint-*/trainer_state.json"))

if state_files:
    with open(state_files[-1]) as f:
        state = json.load(f)

    history      = state.get("log_history", [])
    train_steps  = [h["step"] for h in history if "loss" in h and "eval_loss" not in h]
    train_losses = [h["loss"]  for h in history if "loss" in h and "eval_loss" not in h]
    eval_epochs  = [h["epoch"]     for h in history if "eval_loss" in h]
    eval_losses  = [h["eval_loss"] for h in history if "eval_loss" in h]
    eval_ppls    = [math.exp(l)    for l in eval_losses]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Falcon-7B Fine-Tuning — Training Curves", fontsize=14, fontweight="bold")

    axes[0].plot(train_steps, train_losses, label="Train Loss", color="#2196F3", alpha=0.8)
    if eval_epochs:
        spe = max(train_steps) / max(eval_epochs)
        axes[0].plot([e * spe for e in eval_epochs], eval_losses,
                     label="Eval Loss", color="#F44336", marker="o", lw=2)
    axes[0].set(xlabel="Step", ylabel="Loss", title="Training & Eval Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    if eval_ppls:
        axes[1].plot(eval_epochs, eval_ppls, color="#4CAF50", marker="o", lw=2)
        axes[1].set(xlabel="Epoch", ylabel="Perplexity", title="Eval Perplexity per Epoch")
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plot_path = Path("/kaggle/working/training_curves.png") if ON_KAGGLE else Path("outputs/training_curves.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Plot saved → {plot_path}")
else:
    print("⚠️  No checkpoint found")

---
## Step 10 — Download Artifacts from Kaggle

> Everything is already in `/kaggle/working/`. Use the **Kaggle right-panel Output tab** to download,
> OR run the cell below to create a zip.

In [ ]:
import shutil
from pathlib import Path

WD = Path("/kaggle/working") if ON_KAGGLE else Path("outputs")

print("📋 /kaggle/working/ contents:")
total_mb = 0
for p in sorted(WD.iterdir()):
    if p.name.startswith("."):
        continue
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**2 if p.is_dir() else p.stat().st_size / 1024**2
    total_mb += size
    print(f"  {'📁' if p.is_dir() else '📄'} {p.name:<30} {size:>8.1f} MB")
print(f"\n  Total: {total_mb:.1f} MB")

# Zip just the LoRA adapter (small, ~32 MB — easy to download)
peft_dir = WD / "peft_adapter"
if peft_dir.exists():
    zip_out = str(WD / "peft_adapter")
    shutil.make_archive(zip_out, "zip", str(WD), "peft_adapter")
    size = Path(zip_out + ".zip").stat().st_size / 1024**2
    print(f"\n📦 peft_adapter.zip → {size:.1f} MB")
    print("   Download from Kaggle Output panel on the right sidebar →")
else:
    print("⚠️  peft_adapter not found — training may not have completed")

free_memory()